# Exponential integrators in NGSolve — Part 1: exponential Euler with Krylov subspaces

First of four notebooks on exponential integrators: Part 2 compares methods for $e^{-tA}v$, Part 3 treats non-symmetric operators and time-dependent sources, Part 4 the nonlinear Allen–Cahn equation. Prerequisite: i-tutorial [3.1 Parabolic model problem](https://docu.ngsolve.org/latest/i-tutorials/unit-3.1-parabolic/parabolic.html).

The finite element heat equation $M\dot u + Ku = f$ with $A := M^{-1}K$ has the **exact** time step

$$
u(t_{n+1}) = e^{-\Delta t A}\, u(t_n) + \Delta t\, \varphi_1(-\Delta t A)\, M^{-1} f,
\qquad \varphi_1(z) = \frac{e^z - 1}{z}.
$$

Classical schemes replace $e^{-z}$ by a rational function — $(1+z)^{-1}$ for implicit Euler — and pay with a time discretisation error. Exponential integrators apply $e^{-\Delta tA}$ and $\varphi_1$ to a vector *exactly up to a tolerance*, never forming the dense matrix but using Krylov subspaces with a few sparse solves. The only NGSolve ingredients are `MultiVector.AppendOrthogonalize` and a sparse factorisation.

In [ ]:
from ngsolve import *
from ngsolve.webgui import Draw
import numpy as np
import scipy.linalg as sla
import scipy.sparse as sp
import matplotlib.pyplot as plt
from time import time as walltime
ngsglobals.msg_level = 0

## Model problem

Unit square, Dirichlet boundary, order 3. Mass and stiffness matrices are assembled separately with `symmetric=False`: same sparsity pattern for linear combinations $M + \gamma K$, and a complete `COO()` export to `scipy`.

In [ ]:
mesh = Mesh(unit_square.GenerateMesh(maxh=0.07))
fes = H1(mesh, order=3, dirichlet=".*")
u, v = fes.TnT()

a = BilinearForm(grad(u)*grad(v)*dx, symmetric=False).Assemble()   # stiffness K
m = BilinearForm(u*v*dx, symmetric=False).Assemble()               # mass M
K, M = a.mat, m.mat

f = LinearForm(fes)
f += 1*v*dx                                                        # constant source
f.Assemble()

freedofs = fes.FreeDofs()
P = Projector(freedofs, True)     # sets the Dirichlet dofs of a vector to zero
print(f"ndof = {fes.ndof},  free dofs = {sum(freedofs)}")

Rough initial datum (Gaussian plus a jump), projected onto the free dofs. **Every vector that enters a Krylov method must vanish on the Dirichlet dofs**, otherwise the $M$-inner product couples it to the boundary and the iteration diverges.

In [ ]:
u0 = exp(-100*((x-0.3)**2 + (y-0.6)**2)) + IfPos((x-0.5)*(0.8-x), 1, 0)*IfPos((y-0.2)*(0.5-y), 1, 0)
gfu = GridFunction(fes)
gfu.Set(u0)
gfu.vec.data = P * gfu.vec
u0vec = gfu.vec.CreateVector(); u0vec.data = gfu.vec      # keep a copy of the initial datum
Draw(gfu, mesh, "u0");

## Reference: the $\theta$-scheme

$$
(M + \theta \Delta t K)\, u^{n+1} = (M - (1-\theta)\Delta t K)\, u^n + \Delta t\, f,
\qquad \theta = 1 \text{ (implicit Euler)},\quad \theta = \tfrac12 \text{ (Crank–Nicolson)},
$$

with one factorisation of $M^\ast = M + \theta\Delta t K$, built with the `AsVector` trick of tutorial 3.1.

In [ ]:
def ThetaScheme(theta, dt):
    "returns a function performing one step of the theta-scheme"
    mstar = M.CreateMatrix()
    mstar.AsVector().data = M.AsVector() + theta*dt * K.AsVector()
    inv = mstar.Inverse(freedofs, inverse="sparsecholesky")
    def step(uvec):
        return (inv * (M*uvec - (1-theta)*dt*(K*uvec) + dt*f.vec)).Evaluate()
    return step

## Exact solution and stiffness

$A = M^{-1}K$ is self-adjoint in the $M$-inner product $(u,v)_M = v^T M u$, since $(Au, v)_M = v^T K u = (u, Av)_M$. With the $M$-orthonormal eigenvectors $K w_k = \lambda_k M w_k$, $W^T M W = I$,

$$
e^{-\Delta t A} v = W\, \mathrm{diag}(e^{-\Delta t \lambda_k})\, W^T M v .
$$

On this coarse mesh we compute the dense eigenproblem — **as a reference only**. Errors are measured in $\|v\|_M = \sqrt{v^TMv}$, the $L^2$ norm.

In [ ]:
idx = np.array([i for i in range(fes.ndof) if freedofs[i]])       # free dofs

def dense_free(mat):
    "dense matrix restricted to the free dofs (needs symmetric=False assembly!)"
    r, c, val = mat.COO()
    return sp.csr_matrix((val, (r, c)), shape=(mat.height, mat.width)).toarray()[np.ix_(idx, idx)]

Kd, Md = dense_free(K), dense_free(M)
lam, W = sla.eigh(Kd, Md)               # K W = M W diag(lam),  W^T M W = I
print(f"lambda_min = {lam[0]:.3f}  (exact 2 pi^2 = {2*pi**2:.3f}),   lambda_max = {lam[-1]:.3e}")

def to_np(vec):           return np.array(vec.FV().NumPy())[idx]
def norm_M(vnp):          return sqrt(vnp @ (Md @ vnp))
def exact_funm(func, vnp): return W @ (func(lam) * (W.T @ (Md @ vnp)))       # f(A) v
def relerr(vec, ref_np):  return norm_M(to_np(vec) - ref_np) / norm_M(ref_np)

The *stiffness ratio* $\Delta t\,\lambda_{\max}$ is in the hundreds: the highest modes are damped to zero, the lowest only by $e^{-0.2}$.

In [ ]:
dt = 0.01
print(f"dt*lambda_max = {dt*lam[-1]:.0f},   dt*lambda_min = {dt*lam[0]:.3f}")
u0np  = to_np(u0vec)
ref_exp = exact_funm(lambda l: np.exp(-dt*l), u0np)       # exact e^{-dt A} u0

## Krylov approximation of $f(A)\,v$

Lanczos builds an $M$-orthonormal basis $V_m$ of $\mathcal K_m(A, v) = \mathrm{span}\{v, Av, \dots, A^{m-1}v\}$ together with the small tridiagonal matrix $T_m = V_m^T M A V_m$. The approximation (Saad 1992)

$$
f(A)\, v \;\approx\; \beta\, V_m\, f(T_m)\, e_1, \qquad \beta = \|v\|_M ,
$$

is exact for polynomials of degree $< m$; $f(T_m)$ is evaluated through the eigendecomposition of $T_m$ (never through `inv`).

`MultiVector.AppendOrthogonalize(w, ipmat=M)` is exactly one Lanczos step: it $M$-orthogonalises $w = Av_j$ against the basis (two Gram–Schmidt passes), appends the normalised remainder and **returns the coefficients** — one column of $T_m$. We stop when the approximation changes by less than `tol` between two Krylov dimensions.

In [ ]:
from math import factorial

def phi(k, z):
    "phi_k(z) for numpy arrays, stable near 0:  phi_0 = exp,  phi_{k+1}(z) = (phi_k(z) - 1/k!)/z"
    z = np.asarray(z); out = np.zeros(z.shape, dtype=np.result_type(z, float))
    with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
        small = np.abs(z) < 0.5
        s = np.zeros_like(z[small], dtype=out.dtype)
        for j in range(19, -1, -1):                       # Taylor series sum_j z^j/(j+k)!
            s = s*z[small] + 1.0/factorial(j+k)
        out[small] = s
        p = np.exp(z[~small])
        for i in range(k):
            p = (p - 1.0/factorial(i)) / z[~small]
        out[~small] = p
    return np.where(np.isfinite(out), out, 0.0)         # z = -inf -> 0

def funm_small_sym(T, func):
    "func(T) e_1 for a small symmetric matrix T via its eigendecomposition"
    th, Q = sla.eigh(0.5*(T + T.T))
    return Q @ (func(th) * Q[0, :])

def krylov_funm(op, M, v, func, maxm=60, tol=1e-8, info=None):
    "approximate func(op) v by Lanczos in the M-inner product; op: BaseVector -> BaseVector, M-selfadjoint"
    beta = sqrt(InnerProduct(v, M*v))
    if beta == 0.0:
        return (0.0*v).Evaluate()
    V = MultiVector(v, 0)
    V.Append((1.0/beta) * v)
    T = np.zeros((maxm+1, maxm))
    Eold, hist = np.zeros(0), []
    for j in range(maxm):
        h = np.array(V.AppendOrthogonalize(op(V[j]), ipmat=M)).ravel()   # one Lanczos step
        T[:j+2, j] = h[:j+2]
        E = beta * funm_small_sym(T[:j+1, :j+1], func)                    # coefficients w.r.t. V
        diff = np.linalg.norm(E - np.append(Eold, np.zeros(j+1-len(Eold))))
        hist.append(diff); Eold = E
        if h[j+1] < 1e-14*abs(h).max() or (j >= 1 and diff < tol*beta):   # invariant subspace / converged
            break
    m = len(E)
    if info is not None:
        info["m"], info["hist"] = m, hist
    return (V[0:m] * Vector(E)).Evaluate()

### (a) Polynomial Krylov, $A = M^{-1}K$

One product with $K$ and one mass matrix solve per Lanczos step.

In [ ]:
invM = M.Inverse(freedofs, inverse="sparsecholesky")
tmp = M.CreateColVector()
def opA(z):                                # z -> M^{-1} K z
    out = M.CreateColVector(); tmp.data = K*z; out.data = invM*tmp; return out

ms_poly, errs_poly = [10, 20, 40, 60, 80, 100], []
for mm in ms_poly:
    yk = krylov_funm(opA, M, u0vec, lambda th: np.exp(-dt*th), maxm=mm, tol=0)
    errs_poly.append(relerr(yk, ref_exp))
    print(f"polynomial Krylov  m = {mm:3d}   rel. error = {errs_poly[-1]:.2e}")

Convergence starts only after $m \approx \sqrt{\Delta t\,\lambda_{\max}}$ steps (Hochbruck–Lubich 1997), so it deteriorates like $h^{-1}$ under mesh refinement.

### (b) Shift-and-invert Krylov

Build the Krylov space with the resolvent $B = (M + \gamma K)^{-1}M = (I + \gamma A)^{-1}$ instead (van den Eshof–Hochbruck 2006). $B$ has the same eigenvectors as $A$ and eigenvalues $\theta = 1/(1+\gamma\lambda) \in (0, 1]$: all stiff modes are compressed towards $\theta = 0$. The same Lanczos process with $B$ gives

$$
e^{-\Delta t A} v \approx \beta V_m\, g(\tilde T_m)\, e_1, \qquad g(\theta) = \exp\Big(-\Delta t\, \frac{1/\theta - 1}{\gamma}\Big).
$$

Each step costs one back-substitution with the factorisation of $M + \gamma K$ — the implicit Euler matrix. Convergence is independent of $h$ and $\Delta t$; $\gamma \approx \Delta t/10$ is a good default.

In [ ]:
gamma = 0.1*dt
mstar = M.CreateMatrix()
mstar.AsVector().data = M.AsVector() + gamma*K.AsVector()
invmstar = mstar.Inverse(freedofs, inverse="sparsecholesky")
def opB(z):                                # z -> (M + gamma K)^{-1} M z
    out = M.CreateColVector(); tmp.data = M*z; out.data = invmstar*tmp; return out

def g_exp(th):                             # exp(-dt lambda) with lambda = (1/theta - 1)/gamma
    with np.errstate(divide="ignore"):
        return np.exp(-dt * np.where(th > 0, (1/th - 1)/gamma, np.inf))

ms_si, errs_si = [4, 8, 12, 16, 20, 24], []
for mm in ms_si:
    yk = krylov_funm(opB, M, u0vec, g_exp, maxm=mm, tol=0)
    errs_si.append(relerr(yk, ref_exp))
    print(f"shift-and-invert Krylov  m = {mm:3d}   rel. error = {errs_si[-1]:.2e}")

The built-in stopping estimate (change between consecutive iterates) tracks the true error:

In [ ]:
info = {}
yk = krylov_funm(opB, M, u0vec, g_exp, maxm=40, tol=1e-12, info=info)
true_err = [relerr(krylov_funm(opB, M, u0vec, g_exp, maxm=mm, tol=0), ref_exp) for mm in range(1, info["m"]+1)]

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].semilogy(ms_poly, errs_poly, "o-", label="polynomial Krylov, $A=M^{-1}K$")
ax[0].semilogy(ms_si, errs_si, "s-", label=r"shift-and-invert, $B=(M+\gamma K)^{-1}M$")
ax[0].set_xlabel("Krylov dimension m"); ax[0].set_ylabel(r"rel. error in $\|\cdot\|_M$"); ax[0].legend(); ax[0].grid()
ax[0].set_title(rf"$e^{{-\Delta t A}}u_0$,  $\Delta t\,\lambda_{{max}}$ = {dt*lam[-1]:.0f}")
ax[1].semilogy(range(1, info["m"]+1), true_err, "s-", label="true error")
ax[1].semilogy(range(1, info["m"]+1), np.array(info["hist"])/sqrt(InnerProduct(u0vec, M*u0vec)), "x--", label="estimate (change of iterate)")
ax[1].set_xlabel("Krylov dimension m"); ax[1].legend(); ax[1].grid(); ax[1].set_title("shift-and-invert: error estimator")
plt.tight_layout(); plt.show()

## The exponential Euler method

$$
u^{n+1} = e^{-\Delta t A} u^n + \Delta t\, \varphi_1(-\Delta t A)\, M^{-1} f .
$$

The $\varphi_1$-term comes from the same Krylov routine with a different scalar function; for constant $f$ it is computed once. Wrapping everything in a `BaseMatrix` gives the usual syntax `E * u` (the class is also in `expint_tools.KrylovExp`, with polynomial and Arnoldi variants).

In [ ]:
class KrylovExp(BaseMatrix):
    "y = exp(-dt A) x,  A = M^{-1} K on the free dofs, by shift-and-invert Lanczos with B = (M + gamma K)^{-1} M"
    def __init__(self, M, K, dt, freedofs, gamma=None, tol=1e-8, maxm=60):
        super().__init__()
        self.M, self.dt, self.tol, self.maxm, self.info = M, dt, tol, maxm, {}
        self.gamma = 0.1*dt if gamma is None else gamma
        self.P = Projector(freedofs, True)
        mstar = M.CreateMatrix()
        mstar.AsVector().data = M.AsVector() + self.gamma*K.AsVector()
        self.inv = mstar.Inverse(freedofs, inverse="sparsecholesky")
        self.tmp = M.CreateColVector()

    def Height(self):         return self.M.height
    def Width(self):          return self.M.width
    def IsComplex(self):      return False
    def CreateRowVector(self): return self.M.CreateRowVector()
    def CreateColVector(self): return self.M.CreateColVector()

    def op(self, z):                                    # z -> B z
        out = self.M.CreateColVector(); self.tmp.data = self.M*z; out.data = self.inv*self.tmp; return out

    def Apply(self, x, func):                           # func(A) x  for a scalar function func(lambda)
        def g(th):
            with np.errstate(divide="ignore", over="ignore", invalid="ignore"):
                return func(np.where(th > 0, (1/th - 1)/self.gamma, np.inf))
        xp = self.M.CreateColVector(); xp.data = self.P*x
        return krylov_funm(self.op, self.M, xp, g, self.maxm, self.tol, self.info)

    def Mult(self, x, y):   y.data = self.Apply(x, lambda lam: np.exp(-self.dt*lam))
    def Phi(self, k, x):    return self.Apply(x, lambda lam: phi(k, -self.dt*lam))

The time loop has the same shape as in tutorial 3.1:

In [ ]:
def ExpEuler(dt, tend, tol=1e-8, nsamples=20):
    E = KrylovExp(M, K, dt, freedofs, tol=tol)
    g = dt * E.Phi(1, invM*f.vec)                     # constant source: once
    gfu.Set(u0); gfu.vec.data = P*gfu.vec
    gfut = GridFunction(fes, multidim=0); gfut.AddMultiDimComponent(gfu.vec)
    nsteps = int(round(tend/dt)); ms = []
    for n in range(nsteps):
        gfu.vec.data = E*gfu.vec + g
        ms.append(E.info["m"])
        if (n+1) % max(1, nsteps//nsamples) == 0:
            gfut.AddMultiDimComponent(gfu.vec)
    print(f"{nsteps} steps of size {dt}: Krylov dimension per step between {min(ms)} and {max(ms)}")
    return gfut

t0 = walltime()
gfut = ExpEuler(dt=0.01, tend=0.5)
print(f"wall time {walltime()-t0:.2f} s")
Draw(gfut, mesh, interpolate_multidim=True, animate=True, min=0, max=0.4);

## Accuracy versus $\Delta t$

Smooth compatible data $u_0 = f = \sin(\pi x)\sin(\pi y)$; the exact semi-discrete solution at $T = 0.2$ comes from the eigendecomposition.

In [ ]:
fs = LinearForm(fes); fs += sin(pi*x)*sin(pi*y)*v*dx; fs.Assemble()
gfs = GridFunction(fes); gfs.Set(sin(pi*x)*sin(pi*y)); gfs.vec.data = P*gfs.vec
Tend = 0.2
exact_T = exact_funm(lambda l: np.exp(-Tend*l), to_np(gfs.vec)) + W @ ((1 - np.exp(-Tend*lam))/lam * (W.T @ to_np(fs.vec)))

def run_theta(theta, dt, uinit, fvec):
    mstar = M.CreateMatrix(); mstar.AsVector().data = M.AsVector() + theta*dt*K.AsVector()
    inv = mstar.Inverse(freedofs, inverse="sparsecholesky")
    w = uinit.CreateVector(); w.data = uinit
    for n in range(int(round(Tend/dt))):
        w.data = inv * (M*w - (1-theta)*dt*(K*w) + dt*fvec)
    return w

def run_expeuler(dt, uinit, fvec, tol=1e-8):
    E = KrylovExp(M, K, dt, freedofs, tol=tol)
    g = dt * E.Phi(1, invM*fvec)
    w = uinit.CreateVector(); w.data = uinit
    for n in range(int(round(Tend/dt))):
        w.data = E*w + g
    return w

dts = [0.04, 0.02, 0.01, 0.005, 0.0025]
err = {"implicit Euler": [], "Crank-Nicolson": [], "exponential Euler": []}
for dt_ in dts:
    err["implicit Euler"].append(relerr(run_theta(1.0, dt_, gfs.vec, fs.vec), exact_T))
    err["Crank-Nicolson"].append(relerr(run_theta(0.5, dt_, gfs.vec, fs.vec), exact_T))
    err["exponential Euler"].append(relerr(run_expeuler(dt_, gfs.vec, fs.vec), exact_T))

print(f"{'dt':>8s} " + " ".join(f"{k:>18s}" for k in err))
for i, dt_ in enumerate(dts):
    print(f"{dt_:8.4f} " + " ".join(f"{err[k][i]:18.2e}" for k in err))
for k in ["implicit Euler", "Crank-Nicolson"]:
    print(f"estimated order {k}: {np.log2(err[k][-2]/err[k][-1]):.2f}")

plt.figure(figsize=(5.5, 3.8))
for k, e in err.items(): plt.loglog(dts, e, "o-", label=k)
plt.loglog(dts, [d/dts[0]*err["implicit Euler"][0] for d in dts], "k:", lw=.8, label=r"$O(\Delta t)$")
plt.loglog(dts, [(d/dts[0])**2*err["Crank-Nicolson"][0] for d in dts], "k--", lw=.8, label=r"$O(\Delta t^2)$")
plt.xlabel(r"$\Delta t$"); plt.ylabel("rel. error at T=0.2"); plt.legend(); plt.grid(); plt.show()

The exponential Euler error is independent of $\Delta t$: **the time discretisation error is gone**, and a single step of size $T$ would do.

**Rough data.** Crank–Nicolson is A-stable, but $R(z) = (1+z/2)/(1-z/2) \to -1$ for $z \to -\infty$: unresolved modes are not damped. Exponential integrators inherit the exact damping $e^{-\Delta t\lambda}$.

In [ ]:
exact_T_rough = exact_funm(lambda l: np.exp(-Tend*l), u0np) + W @ ((1 - np.exp(-Tend*lam))/lam * (W.T @ to_np(f.vec)))
print(f"{'dt':>8s} {'implicit Euler':>16s} {'Crank-Nicolson':>16s} {'exp. Euler':>16s}")
for dt_ in dts:
    e1 = relerr(run_theta(1.0, dt_, u0vec, f.vec), exact_T_rough)
    e2 = relerr(run_theta(0.5, dt_, u0vec, f.vec), exact_T_rough)
    e3 = relerr(run_expeuler(dt_, u0vec, f.vec), exact_T_rough)
    print(f"{dt_:8.4f} {e1:16.2e} {e2:16.2e} {e3:16.2e}")

## Cost

Per step: $m \approx 15$–$30$ back-substitutions instead of one — but a single step of any size gives an accuracy that implicit Euler needs thousands of steps for.

In [ ]:
t0 = walltime()
E = KrylovExp(M, K, 0.5, freedofs, tol=1e-10)
uT = (E*u0vec + 0.5*E.Phi(1, invM*f.vec)).Evaluate()
t1 = walltime() - t0
ex05 = exact_funm(lambda l: np.exp(-0.5*l), u0np) + W @ ((1 - np.exp(-0.5*lam))/lam * (W.T @ to_np(f.vec)))
print(f"one exponential Euler step to T=0.5:  {t1:.3f} s,  Krylov dim {E.info['m']},  rel. error {relerr(uT, ex05):.1e}")

for dt_, nsteps in [(1e-2, 50), (1e-3, 500), (1e-4, 5000)]:
    t0 = walltime()
    step = ThetaScheme(1.0, dt_); w = u0vec.CreateVector(); w.data = u0vec
    for n in range(nsteps): w.data = step(w)
    print(f"{nsteps:5d} implicit Euler steps (dt={dt_:.0e}) to T=0.5:  {walltime()-t0:.3f} s,  rel. error {relerr(w, ex05):.1e}")

## Tests

In [ ]:
# (i) a single eigenmode decays like exp(-2 pi^2 t); the error is the p=3 discretisation error
gfe = GridFunction(fes); gfe.Set(sin(pi*x)*sin(pi*y)); gfe.vec.data = P*gfe.vec
E = KrylovExp(M, K, 0.01, freedofs, tol=1e-10)
for n in range(10): gfe.vec.data = E*gfe.vec
err_mode = sqrt(Integrate((gfe - exp(-2*pi**2*0.1)*sin(pi*x)*sin(pi*y))**2, mesh))
print(f"eigenmode test: L2 error {err_mode:.2e},  Krylov dim {E.info['m']}")
assert err_mode < 1e-5 and E.info["m"] <= 6

# (ii) rough data: Krylov approximations converge to the dense reference
assert errs_poly[-1] < 1e-6 and errs_si[-1] < 1e-8 and min(true_err) < 1e-10

# (iii) phi_1 against the dense reference
Minvf = (invM*f.vec).Evaluate()
g1 = KrylovExp(M, K, dt, freedofs, tol=1e-10).Phi(1, Minvf)
assert relerr(g1, exact_funm(lambda l: phi(1, -dt*l), to_np(Minvf))) < 1e-8

# (iv) orders of the time stepping methods
assert 0.8 < np.log2(err["implicit Euler"][-2]/err["implicit Euler"][-1]) < 1.2
assert 1.8 < np.log2(err["Crank-Nicolson"][-2]/err["Crank-Nicolson"][-1]) < 2.2
assert max(err["exponential Euler"]) < 1e-6
print("all tests passed")

## Summary

* Exact step $u^{n+1} = e^{-\Delta tA}u^n + \Delta t\varphi_1(-\Delta tA)M^{-1}f$; $A$ is self-adjoint in the $M$-inner product, so Krylov methods use `AppendOrthogonalize(..., ipmat=M)`.
* Polynomial Krylov needs $m \sim \sqrt{\Delta t\lambda_{\max}}$ steps; **shift-and-invert Krylov** with one factorisation of $M + \gamma K$ converges in 10–30 steps, independently of the mesh.
* $\varphi$-functions: same routine, different scalar function.

**References:** Y. Saad, SIAM J. Numer. Anal. 29 (1992); M. Hochbruck, C. Lubich, SIAM J. Numer. Anal. 34 (1997); J. van den Eshof, M. Hochbruck, SIAM J. Sci. Comput. 27 (2006); M. Hochbruck, A. Ostermann, Acta Numerica 19 (2010). Full list with DOIs in the README.